# Noisy warm-start experiment: optimizer comparison

The canonical experiment (`experiment.py`), now with **noise on**: the full Starmon-7 noise model (T1/T2 decay during gates and idling, single-qubit and CZ depolarization, readout relaxation and assignment error) instead of the ideal simulator.

**Protocol (one chain).**
1. Start at the **known β → 0 point**: the Gibbs state is maximally mixed, and the Bell-pair prefix at near-zero angles already prepares it exactly (see `beta_zero_sanity_check.ipynb`). Initial angles `x0 = [0.1, 0.1, 0.1, 0.1]`, the same as `experiment.py`.
2. Optimize the shot-based cost (`CostFunction`, 8192 shots, readout-corrected) at each β in `BETAS` in turn, **warm-starting** each step from the previous step's angles.
3. After each step, reconstruct ρ_A by 9-setting state tomography and compute its fidelity to the exact Gibbs state.

**Comparison.** The same chain is repeated for each optimizer, `N_REPEATS` times each (shot noise makes every run different):

| Optimizer | Why it's here |
|---|---|
| **COBYLA** | the baseline: what `experiment.py` and the GUI use (`rhobeg=0.5`) |
| **Nelder-Mead** | derivative-free simplex. Given a 0.5 rad initial simplex, since scipy's default (5% of x0) is too small to move under shot noise |
| **Powell** | derivative-free, line searches along each direction |
| **SPSA** | built for noisy objectives: 2 evaluations per step whatever the dimension. The standard choice for variational algorithms on hardware |

**Fair budget.** Every optimizer gets exactly `MAX_EVALS` cost evaluations (circuit executions) per β step, enforced by a hard counter. If the cap trips mid-iteration (e.g. inside a Powell line search), the lowest-cost point measured so far is used.

**Reference points** for each β, independent of any optimizer:
- *noiseless ceiling*: fidelity of the best angles under the exact noiseless cost. This is the limit of the circuit and cost themselves (~0.99, not 1, at β > 0).
- *noisy ceiling*: those same best angles, run through noisy tomography. Roughly the best this hardware can show at that β, with a perfect optimizer.

**Runtime.** One noisy cost evaluation takes ~3–5 s of single-threaded simulation, so all chains run in parallel worker processes. With the defaults below (4 optimizers × 3 repeats × 4 β × 80 evals), expect ~30–45 min on a many-core machine. Results are cached in `results/`, and re-running the notebook only replots unless you change the settings or set `RERUN = True`.

In [ ]:
import os
import sys
import pickle
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

# Works whether the kernel starts in experiments/ or in the repo root.
HERE = Path.cwd()
EXP_DIR = next(p for p in (HERE, HERE / "experiments", HERE / "gibbs_tfd_reproduction" / "experiments")
               if (p / "noisy_optimizer_benchmark.py").exists())
sys.path.insert(0, str(EXP_DIR))

import noisy_optimizer_benchmark as bench

PLOTS_DIR = EXP_DIR / "plots"
RESULTS_DIR = EXP_DIR / "results"
PLOTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

## Settings

In [ ]:
BETAS = [1e-8, 0.5, 1, 5]          # canonical schedule; 1e-8 stands in for beta = 0 (beta**-varsigma is singular at 0)
X0 = [0.1, 0.1, 0.1, 0.1]          # near the known beta -> 0 solution, as in experiment.py
OPTIMIZERS = list(bench.OPTIMIZERS)  # COBYLA, Nelder-Mead, Powell, SPSA
MAX_EVALS = 80                     # cost evaluations per beta step, identical for every optimizer
N_REPEATS = 3                      # independent chains per optimizer
TOMO_SHOTS = 8192
SEED = 0                           # seeds SPSA's random perturbations (shot noise itself is unseeded)
N_WORKERS = max(1, (os.cpu_count() or 2) - 2)
RERUN = False                      # True: ignore cached results and run again

RESULTS_FILE = RESULTS_DIR / "noisy_optimizer_comparison.pkl"

## Run (or load cached results)

Each (optimizer, repeat) chain and each β's reference point is one job in the process pool.

In [ ]:
config = dict(betas=BETAS, x0=X0, optimizers=OPTIMIZERS, max_evals=MAX_EVALS,
              n_repeats=N_REPEATS, tomo_shots=TOMO_SHOTS, seed=SEED)

cached = None
if RESULTS_FILE.exists() and not RERUN:
    with open(RESULTS_FILE, "rb") as f:
        cached = pickle.load(f)
    if cached.get("config") != config:
        print("Cached results were made with different settings -- running again.")
        cached = None

if cached is not None:
    chains, references = cached["chains"], cached["references"]
    print(f"Loaded {len(chains)} chains from {RESULTS_FILE.name} (set RERUN = True to redo).")
else:
    t0 = time.time()
    chains, references = [], {}
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        chain_jobs = {pool.submit(bench.run_chain, opt, rep, BETAS, X0, MAX_EVALS, TOMO_SHOTS, SEED): (opt, rep)
                      for opt in OPTIMIZERS for rep in range(N_REPEATS)}
        ref_jobs = {pool.submit(bench.reference_point, beta, TOMO_SHOTS): beta for beta in BETAS}
        for fut in as_completed(list(chain_jobs) + list(ref_jobs)):
            if fut in ref_jobs:
                references[ref_jobs[fut]] = fut.result()
                print(f"[{time.time() - t0:6.0f}s] reference beta={ref_jobs[fut]} done")
            else:
                chain = fut.result()
                chains.append(chain)
                fids = "  ".join(f"{s['fidelity']:.3f}" for s in chain["steps"])
                print(f"[{time.time() - t0:6.0f}s] {chain['optimizer']:<11} repeat {chain['repeat']}  fidelities per beta: {fids}")
    chains.sort(key=lambda c: (OPTIMIZERS.index(c["optimizer"]), c["repeat"]))
    with open(RESULTS_FILE, "wb") as f:
        pickle.dump({"config": config, "chains": chains, "references": references}, f)
    print(f"Done in {(time.time() - t0) / 60:.1f} min -- saved to {RESULTS_FILE.name}")

## Results table

One row per (optimizer, repeat, β).
- `fidelity`: measured under noise, from tomography.
- `fidelity_ideal_angles`: the same final angles evaluated without noise. This isolates how good the angles the optimizer found are, separate from what noise does to the state.

In [ ]:
rows = []
for chain in chains:
    for s in chain["steps"]:
        rows.append({
            "optimizer": chain["optimizer"], "repeat": chain["repeat"], "beta": s["beta"],
            "fidelity": s["fidelity"], "fidelity_ideal_angles": s["fidelity_ideal_angles"],
            "n_evals": s["n_evals"], "minutes": s["seconds"] / 60,
            "gamma1": s["params"][0], "gamma2": s["params"][1],
            "alpha1": s["params"][2], "alpha2": s["params"][3],
        })
df = pd.DataFrame(rows)

ref_df = pd.DataFrame([{"beta": b, "noiseless ceiling": r["fidelity_exact"], "noisy ceiling": r["fidelity_noisy"]}
                       for b, r in sorted(references.items())]).set_index("beta")


def mean_std(x):
    return f"{x.mean():.3f} ± {x.std(ddof=0):.3f}"


summary = df.pivot_table(index="optimizer", columns="beta", values="fidelity", aggfunc=mean_std).reindex(OPTIMIZERS)
summary_ideal = df.pivot_table(index="optimizer", columns="beta", values="fidelity_ideal_angles",
                               aggfunc=mean_std).reindex(OPTIMIZERS)
print("Measured fidelity under noise (mean ± std over repeats):")
display(summary)
print("Fidelity of the found angles without noise (optimizer quality):")
display(summary_ideal)
print("Reference points:")
display(ref_df.T.round(4))

## Figures

Colors identify optimizers the same way in every figure, and each optimizer also has its own marker shape.

In [ ]:
# Validated categorical palette (slots 1-4); grays for references and ink.
COLORS = {"COBYLA": "#2a78d6", "Nelder-Mead": "#eb6834", "Powell": "#1baf7a", "SPSA": "#eda100"}
MARKERS = {"COBYLA": "o", "Nelder-Mead": "s", "Powell": "^", "SPSA": "D"}
INK, INK_DIM, GRID = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": INK_DIM, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK_DIM, "ytick.color": INK_DIM,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "lines.linewidth": 2, "font.size": 10, "axes.titlesize": 11, "legend.frameon": False,
})


def beta_label(b):
    return "β → 0" if b < 1e-6 else f"β = {b:g}"


def optimizer_legend(fig, extra=(), y=1.0):
    handles = [Line2D([], [], color=COLORS[o], marker=MARKERS[o], linestyle="-", markersize=7, label=o)
               for o in OPTIMIZERS] + list(extra)
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, y), ncol=len(handles), fontsize=9)

### 1. Final fidelity per β

**Left:** the found angles evaluated without noise, which shows how well each optimizer did its job. **Right:** what tomography actually measures under noise. Each small marker is one repeat, and the bar is the mean.

- The gap to the *noiseless ceiling* on the left is optimizer error.
- The gap between the two ceilings on the right is the hardware's own cost, which no optimizer can recover.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
x_beta = np.arange(len(BETAS))
width = 0.8 / len(OPTIMIZERS)

panels = [("fidelity_ideal_angles", "Found angles, evaluated without noise"),
          ("fidelity", "Measured under noise (tomography)")]
for ax, (col, title) in zip(axes, panels):
    for i, opt in enumerate(OPTIMIZERS):
        xo = x_beta - 0.4 + width * (i + 0.5)
        sub = df[df.optimizer == opt]
        means = [sub[sub.beta == b][col].mean() for b in BETAS]
        ax.hlines(means, xo - width * 0.4, xo + width * 0.4, color=COLORS[opt], linewidth=3)
        for j, b in enumerate(BETAS):
            vals = sub[sub.beta == b][col].values
            ax.scatter(np.full(len(vals), xo[j]), vals, marker=MARKERS[opt], s=26, color=COLORS[opt],
                       edgecolor="#fcfcfb", linewidth=0.8, zorder=3, alpha=0.9)
    for j, b in enumerate(BETAS):
        ref = references[b]
        ax.hlines(ref["fidelity_exact"], j - 0.45, j + 0.45, color=INK_DIM, linestyle="--", linewidth=1.2)
        if col == "fidelity":
            ax.hlines(ref["fidelity_noisy"], j - 0.45, j + 0.45, color=INK_DIM, linestyle=":", linewidth=1.5)
    ax.set_xticks(x_beta, [beta_label(b) for b in BETAS])
    ax.set_title(title, loc="left")
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("fidelity to exact Gibbs state ρ_A")
lo = min(df.fidelity.min(), df.fidelity_ideal_angles.min(), ref_df.values.min())
axes[0].set_ylim(max(0.0, lo - 0.03), 1.005)

optimizer_legend(fig, extra=[
    Line2D([], [], color=INK_DIM, linestyle="--", linewidth=1.2, label="noiseless ceiling"),
    Line2D([], [], color=INK_DIM, linestyle=":", linewidth=1.5, label="noisy ceiling"),
])
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(PLOTS_DIR / "noisy_fidelity_by_optimizer.png", dpi=130, bbox_inches="tight")
plt.show()

### 2. Cost convergence within each β step

The y-axis is the normalized gap to the cost's true minimum, (C − C_min) / |C_min|, where C_min is the lowest eigenvalue of the cost operator. Lower is better, and 0 is only reachable without noise. Faint lines are individual repeats, and the bold line is the median across repeats.

Each β step starts from the previous step's angles. A low starting point means the warm start was already close.

In [ ]:
fig, axes = plt.subplots(1, len(BETAS), figsize=(4 * len(BETAS), 3.8), sharey=False)
for ax, b in zip(np.atleast_1d(axes), BETAS):
    for opt in OPTIMIZERS:
        gaps = []
        for chain in chains:
            if chain["optimizer"] != opt:
                continue
            step = next(s for s in chain["steps"] if s["beta"] == b)
            gap = (step["costs"] - step["lower_bound"]) / abs(step["lower_bound"])
            gaps.append(gap)
            ax.plot(np.arange(1, len(gap) + 1), gap, color=COLORS[opt], linewidth=0.8, alpha=0.25)
        n = min(len(g) for g in gaps)
        med = np.median([g[:n] for g in gaps], axis=0)
        ax.plot(np.arange(1, n + 1), med, color=COLORS[opt], linewidth=2)
        ax.plot(n, med[-1], marker=MARKERS[opt], color=COLORS[opt], markersize=7,
                markeredgecolor="#fcfcfb", markeredgewidth=1.2)
    ax.set_title(beta_label(b), loc="left")
    ax.set_xlabel("cost evaluation")
    ax.set_ylim(bottom=min(0, ax.get_ylim()[0]))
np.atleast_1d(axes)[0].set_ylabel("normalized gap to minimum")
optimizer_legend(fig)
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(PLOTS_DIR / "noisy_cost_convergence.png", dpi=130, bbox_inches="tight")
plt.show()

### 3. Reconstructed ρ_A vs the exact Gibbs state

Real part of ρ_A. For each optimizer, the repeat with the **median** fidelity is shown, so this is a typical run rather than the luckiest. The imaginary parts are near zero, since the target is real.

In [ ]:
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#e34948", "#f0efec", "#2a78d6"])
LABELS = ["00", "01", "10", "11"]


def typical_rho(opt, b):
    sub = df[(df.optimizer == opt) & (df.beta == b)].sort_values("fidelity")
    rep = sub.iloc[len(sub) // 2]
    chain = next(c for c in chains if c["optimizer"] == opt and c["repeat"] == rep["repeat"])
    step = next(s for s in chain["steps"] if s["beta"] == b)
    return step["rho_A"], step["fidelity"]


columns = ["Exact Gibbs"] + OPTIMIZERS
vmax = 0.5
fig, axes = plt.subplots(len(BETAS), len(columns), figsize=(2.3 * len(columns), 2.3 * len(BETAS)))
axes = np.atleast_2d(axes)
for r, b in enumerate(BETAS):
    for c, name in enumerate(columns):
        ax = axes[r, c]
        if name == "Exact Gibbs":
            rho, label = bench.gibbs_target(b), "target"
        else:
            rho, fid = typical_rho(name, b)
            label = f"F = {fid:.3f}"
        im = ax.imshow(np.real(rho), cmap=DIVERGING, vmin=-vmax, vmax=vmax)
        ax.set_xticks(range(4), LABELS, fontsize=7)
        ax.set_yticks(range(4), LABELS, fontsize=7)
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_title(label, fontsize=9, color=INK_DIM)
        if r == 0:
            ax.text(0.5, 1.28, name, transform=ax.transAxes, ha="center", fontsize=11,
                    color=COLORS.get(name, INK), fontweight="bold")
        if c == 0:
            ax.set_ylabel(beta_label(b), fontsize=11)
cbar = fig.colorbar(im, ax=axes, shrink=0.6, pad=0.02)
cbar.set_label("Re ρ_A")
fig.savefig(PLOTS_DIR / "noisy_rho_grid.png", dpi=130, bbox_inches="tight")
plt.show()

### 4. Cost of each run

Wall time per β step, per chain (including the tomography at the end of the step). Every optimizer used the same number of evaluations, so the differences here are mostly simulator variance, not the optimizer.

In [1]:
cost_table = df.groupby(["optimizer", "beta"]).agg(evals=("n_evals", "mean"), minutes=("minutes", "mean"))
display(cost_table.unstack("beta").reindex(OPTIMIZERS).round(1))

NameError: name 'df' is not defined

## Ranking

Optimizers are ranked per β by mean measured fidelity, alongside the gap to the noisy ceiling (the best this hardware can show at that β).

In [ ]:
for b in BETAS:
    sub = df[df.beta == b].groupby("optimizer")["fidelity"].agg(["mean", "std", "min"]).sort_values("mean", ascending=False)
    sub["gap to noisy ceiling"] = references[b]["fidelity_noisy"] - sub["mean"]
    print(f"\n{beta_label(b)}   (noisy ceiling {references[b]['fidelity_noisy']:.3f}, "
          f"noiseless ceiling {references[b]['fidelity_exact']:.3f})")
    print(sub.round(4).to_string())

## Conclusions

*(Fill in after running.)* Questions to answer from the figures:
- **How much does noise cost?** The gap between the two ceilings at each β.
- **Which optimizer handles shot and gate noise best?** Measured fidelity (figure 1, right) and its spread across repeats.
- **Is the remaining gap optimizer error or noise?** If an optimizer's found angles score near the noiseless ceiling (figure 1, left) but its measured fidelity is low, the noise is to blame. If its angles score low too, the optimizer is.
- **Does the warm start hold up as β grows?** Figure 2: does each step start near its final value, or does it have to climb back?